# RUS 1.3 — Qwen2.5 7B on Kaggle dual T4

Enable **Internet** and select the **GPU T4 x2** accelerator. This notebook uses Accelerate model parallelism with `balanced`: transformer blocks are deliberately distributed across both GPUs, with conservative memory caps that leave room for activations and projected fp16 layers.

> Accelerate's device map is sequential model parallelism, not tensor parallelism: both GPUs store model layers, but they do not execute the same layer simultaneously.

In [ ]:
# @title 0. Verify two T4 GPUs and disk space
!nvidia-smi --query-gpu=index,name,memory.total,memory.free --format=csv,noheader
!df -h /kaggle/working
import torch
assert torch.cuda.device_count() == 2, f'Expected Kaggle T4 x2, found {torch.cuda.device_count()} GPU(s)'
for i in range(2):
    assert 'T4' in torch.cuda.get_device_name(i), torch.cuda.get_device_name(i)
print('Dual T4 ready')

In [ ]:
# @title 1. Install dependencies and this branch
!pip install -q --upgrade 'transformers>=4.48,<6' 'accelerate>=0.30,<2' 'bitsandbytes>=0.43' safetensors huggingface_hub
!pip install -q --no-cache-dir --force-reinstall --no-deps git+https://github.com/CodexNexor/rus.git@agent/reliable-ablation-pipeline
import rus, transformers, bitsandbytes, accelerate
print('RUS', rus.__version__, '| transformers', transformers.__version__, '| accelerate', accelerate.__version__, '| bnb', bitsandbytes.__version__)

In [ ]:
# @title 2. Optional Hugging Face authentication from a Kaggle secret
# Add a Kaggle secret named HF_TOKEN for higher Hub rate limits.
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(token=UserSecretsClient().get_secret('HF_TOKEN'), add_to_git_credential=False)
    print('Hugging Face authentication active')
except Exception as exc:
    print('Continuing unauthenticated:', type(exc).__name__)

In [ ]:
# @title 3. Configure the 7B dual-GPU run
MODEL = 'Qwen/Qwen2.5-7B-Instruct'
MAX_MEMORY = {0: '12GiB', 1: '13GiB', 'cpu': '24GiB'}
NUM_PROMPTS = 48
SOURCE_DIRECTIONS = 5
COEFFICIENT = 0.8
OUTPUT_DIR = '/kaggle/working/rus_output'
print(MODEL, MAX_MEMORY)

In [ ]:
# @title 4. Load across both T4 GPUs and verify placement
from rus import RusEngine
engine = RusEngine(
    MODEL, load_in_8bit=True, output_dir=OUTPUT_DIR,
    device_map='balanced', max_memory=MAX_MEMORY,
)
engine.load()
report = engine.device_report()
print(report)
assert report['active_cuda_devices'] == [0, 1], (
    f"Model did not use both GPUs: {report['module_placement']}"
)
print('Both GPUs contain model modules')

In [ ]:
# @title 5. Analyze and apply global norm-preserving ablation
engine.analyze(num_prompts=NUM_PROMPTS, protect_harmless=True)
engine.show_refusal(top_n=20)
engine.ablate(
    k=SOURCE_DIRECTIONS, coefficient=COEFFICIENT,
    strategy='global', preserve_norm=True,
)
reductions = [t['reduction'] for layer in engine.ablation_stats.values() for t in layer['targets'].values()]
print('Consensus sources:', engine.source_layers)
print('Destination layer count:', len(engine.selected_layers))
print('Projection reduction min/mean:', min(reductions), sum(reductions)/len(reductions))
print('Memory after surgery:', engine.device_report())
assert reductions and min(reductions) > 0.5

In [ ]:
# @title 6. Held-out behavioral and KL comparison
results = engine.compare()
print({k: round(v, 4) for k, v in results.items() if isinstance(v, float)})

In [ ]:
# @title 7. Save, release the original allocation, reload, and run a real forward pass
import gc, json, os, torch
path = engine.save()
print('Saved:', path)
del engine
gc.collect(); torch.cuda.empty_cache()
from transformers import AutoModelForCausalLM, AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(path)
model = AutoModelForCausalLM.from_pretrained(
    path, device_map='balanced', max_memory=MAX_MEMORY, low_cpu_mem_usage=True
)
model.eval()
from rus.evaluator import generate_response
probe = generate_response(model, tokenizer, 'State the capital of France.', max_new_tokens=12)
print('Reload + forward OK:', model.__class__.__name__, '|', probe)
meta = json.load(open(os.path.join(path, 'rus_metadata.json')))
assert meta['quantization_skip_modules'], 'Mixed-quantization reload metadata missing'

In [ ]:
# @title 8. Optional private research upload
# from huggingface_hub import HfApi, whoami
# repo_id = f"{whoami()['name']}/Qwen2.5-7B-Instruct-RUS-Research"
# api = HfApi(); api.create_repo(repo_id, private=True, exist_ok=True)
# api.upload_folder(folder_path=path, repo_id=repo_id)
# print('Private checkpoint:', repo_id)